# POC 3: The Budget Variance Narrator

**Pain point:** Every quarter, finance asks cost centre owners to explain their budget variances. People write the same vague paragraph — 'primarily driven by project timing and headcount phasing' — because reconstructing the actual reasons requires digging through 10 project updates. The commentary ends up wrong or so hedged it's useless.

**What this notebook shows:** Grounding with actuals vs. budget data AND context notes produces specific, per-line commentary. The ungrounded model produces the exact vague paragraph described above.

**You need:** A free Groq API key from https://console.groq.com

**Connecting Dots**, is where I write about the patterns I notice while building, checkout my blogs for more

👉 https://sriharshacr.github.io/blogs/

In [ ]:
!pip install groq -q

In [ ]:
import os
from groq import Groq

try:
    from google.colab import userdata
    GROQ_API_KEY = userdata.get('GROQ_API_KEY')
except Exception:
    GROQ_API_KEY = os.environ.get('GROQ_API_KEY') or input('Enter Groq API key: ')

client = Groq(api_key=GROQ_API_KEY)
MODEL = 'openai/gpt-oss-120b'

def call_llm(system_prompt, user_message, temperature=0.3, max_tokens=900):
    response = client.chat.completions.create(
        model=MODEL,
        messages=[
            {'role': 'system', 'content': system_prompt},
            {'role': 'user', 'content': user_message}
        ],
        temperature=temperature,
        max_tokens=max_tokens
    )
    return response.choices[0].message.content

print(f'Model ready: {MODEL}')

In [ ]:
# --- Synthetic grounding material ---
# Replace with your actual budget vs actuals export and project notes.

BUDGET_ACTUALS = """
Q3 2025 — Engineering Cost Centre (CC: ENG-042) — Budget vs Actuals (£000s)

Line item                  | Budget | Actual | Variance | +/- %
---------------------------|--------|--------|----------|---------
External Consultants       |  120   |   78   |   -42    | -35%  (underspend)
Cloud Infrastructure       |   85   |   97   |   +12    | +14%  (overspend)
Training & Development     |   30   |   11   |   -19    | -63%  (underspend)
Software Licences          |   45   |   48   |    +3    |  +7%  (overspend)
Contractor Travel          |   18   |    6   |   -12    | -67%  (underspend)
---------------------------|--------|--------|----------|---------
TOTAL                      |  298   |  240   |   -58    | -19%  (underspend)
"""

CONTEXT_NOTES = """
Project and operational context for Q3:

- Data Platform Project (DPP): originally scoped to start Q3, delayed to Q4 start due to
  procurement approval taking 6 weeks longer than planned. This is the primary driver of
  the external consultants underspend — the 3 consultants booked for DPP weren't onboarded
  until 1 October.

- Cloud infra overspend: an unplanned data migration was approved mid-quarter (30 Jul)
  to decommission the legacy Postgres cluster before year-end. Additional S3 and RDS costs
  of approx £14k; offset by £2k saving from rightsizing 4 idle EC2 instances.

- Training underspend: the Q3 L&D programme (3 x team workshops, 2 x external certs) was
  deferred to Q4 because the product launch in August required all-hands focus.
  Budget is not lost — it is carried into Q4 with Finance approval.

- Software licence overspend: 3 new engineers joined in late August; annual licences for
  GitHub Enterprise and Datadog were prorated for 4 months, totalling £3.2k over budget.

- Contractor travel underspend: planned onsite visits to the Mumbai office (3 trips, £12k)
  were replaced with remote working sessions due to internal travel freeze in July.
"""

FULL_CONTEXT = f"""
{BUDGET_ACTUALS}

=== CONTEXT NOTES ===
{CONTEXT_NOTES}
"""

print('Grounding data loaded.')

In [ ]:
# --- UNGROUNDED call ---
# Model has only the variance numbers — no context about why.

ungrounded_system = "You are a financial analyst. Write clear, professional commentary."

ungrounded_query = f"""
Write Q3 budget variance commentary for the Engineering cost centre.
This will be included in the quarterly finance review pack.

{BUDGET_ACTUALS}
"""

print('=== UNGROUNDED OUTPUT ===')
print(call_llm(ungrounded_system, ungrounded_query))

In [ ]:
# --- GROUNDED call ---
# Model gets the numbers AND the context notes explaining the actual reasons.

grounded_system = """
You are a financial analyst writing variance commentary for a quarterly finance review.

Rules:
- Write one paragraph per material variance line (>£5k or >10% deviation)
- Name the specific cause from the context notes — not a generic phrase
- State whether the variance is permanent or timing (will it reverse in Q4?)
- Flag any variance that creates a Q4 risk or budget pressure
- Do not use the phrase 'driven by project timing' without naming the specific project
- Total commentary length: under 250 words
"""

grounded_query = f"""
Write Q3 budget variance commentary for the Engineering cost centre (CC: ENG-042).
This will be included in the quarterly finance review pack presented to the CFO.

{FULL_CONTEXT}
"""

print('=== GROUNDED OUTPUT ===')
print(call_llm(grounded_system, grounded_query))

In [ ]:
# --- BONUS: Q4 budget pressure flag ---
# Ask a follow-on question without re-loading the full context.
# This demonstrates efficient context reuse.

followup_system = """
You are a financial analyst. Answer based only on the budget data and context provided.
Be specific. Use line items and amounts.
"""

followup_query = f"""
Based on the Q3 variances and context below, which line items are most likely
to create Q4 budget pressure? Rank them and explain why.

{FULL_CONTEXT}
"""

print('=== Q4 BUDGET RISK ANALYSIS ===')
print(call_llm(followup_system, followup_query, max_tokens=400))

## What just happened

The **ungrounded** output is the paragraph everyone already knows and hates: 'The underspend in external consultants reflects timing differences in project commencement, while cloud infrastructure overspend was driven by increased utilisation.' True but useless — it doesn't help the CFO make a decision.

The **grounded** output names the Data Platform Project, the legacy Postgres decommission, the product launch trade-off, and the proration of new joiner licences. It also flags which variances will reverse in Q4 — which is the one thing the CFO actually needs to know.

**Output constraint note:** The grounded system prompt includes `'Do not use the phrase driven by project timing without naming the specific project'` — this is Scenario 3 from the series (output constraint grounding). It prevents the exact failure mode you're trying to avoid.